# ADIA_TTS — premier essai de lecteur wolof

Notebook de test, pas un moteur de production. Il récupère les entrées Wolof–Français déjà présentes dans le dépôt, génère un petit lot de WAV, puis exporte les métadonnées pour contrôle humain.

**Important :** les entrées du dictionnaire sont marquées comme non vérifiées pour la prononciation. Les audios produits sont des candidats, pas des exemples validés. Ne pas les republier avant contrôle linguistique et vérification des licences.

Modèle candidat : [CONCREE/Adia_TTS](https://huggingface.co/CONCREE/Adia_TTS), licence indiquée Apache-2.0. Le modèle est volumineux ; utiliser un runtime GPU gratuit si disponible. Les quotas gratuits peuvent changer.

In [ ]:
%pip -q install git+https://github.com/huggingface/parler-tts.git transformers accelerate soundfile pandas

In [ ]:
import json, os, re, csv, zipfile, requests, torch, soundfile as sf
import pandas as pd
from pathlib import Path
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

MODEL_ID = 'CONCREE/Adia_TTS'
DATA_URL = 'https://raw.githubusercontent.com/aydiarra-star/soninke-tts-libre/main/data/wolof-fr-dictionary.json'
OUT = Path('/content/adia_wolof_samples')
OUT.mkdir(parents=True, exist_ok=True)

response = requests.get(DATA_URL, timeout=60)
response.raise_for_status()
dictionary = response.json()
entries = dictionary.get('entries', [])
samples, seen = [], set()
for entry in entries:
    word = (entry.get('wolof') or {}).get('word', '').strip()
    if not word or word.casefold() in seen:
        continue
    seen.add(word.casefold())
    samples.append({
        'id': entry.get('id', ''),
        'wolof_text': word,
        'fr_text': (entry.get('fr') or {}).get('word', ''),
        'source': entry.get('source', ''),
        'source_status': entry.get('pronunciationStatus', 'unknown'),
    })
    if len(samples) >= 10:
        break

if len(samples) < 1:
    raise RuntimeError('Aucune entrée wolof exploitable trouvée.')
pd.DataFrame(samples)

## Charger le modèle

Le chargement peut prendre plusieurs minutes. Si la mémoire est insuffisante, arrêter proprement et essayer un runtime avec davantage de RAM/GPU ; ne pas modifier le modèle ni ses poids sans consigner les changements.

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Appareil :', device)
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
sampling_rate = int(model.config.sampling_rate)
print('Fréquence audio :', sampling_rate)

In [ ]:
voice_description = 'A clear and educational voice, with a flow adapted to language learning.'
manifest = []
for i, item in enumerate(samples, start=1):
    text = item['wolof_text']
    safe_id = re.sub(r'[^a-zA-Z0-9_-]+', '_', item['id'] or f'word_{i}').strip('_')
    filename = f'{i:02d}_{safe_id}.wav'
    description_ids = tokenizer(voice_description, return_tensors='pt').input_ids.to(device)
    prompt_ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
    with torch.inference_mode():
        audio = model.generate(input_ids=description_ids, prompt_input_ids=prompt_ids)
    audio_array = audio.detach().cpu().numpy().squeeze()
    path = OUT / filename
    sf.write(path, audio_array, sampling_rate, subtype='PCM_16')
    duration = len(audio_array) / sampling_rate if getattr(audio_array, 'ndim', 0) else 0
    manifest.append({**item, 'model_id': MODEL_ID, 'sampling_rate': sampling_rate, 'audio_file': filename, 'duration_seconds': round(duration, 3), 'validation_status': 'unvalidated'})
    print(f'{i}/{len(samples)} — {text} — {duration:.2f} s')

manifest_path = OUT / 'manifest.csv'
pd.DataFrame(manifest).to_csv(manifest_path, index=False, encoding='utf-8')
print('Manifest :', manifest_path)

In [ ]:
zip_path = '/content/adia_wolof_samples.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUT.iterdir():
        if path.is_file():
            zf.write(path, arcname=path.name)
print('Archive prête :', zip_path)

from google.colab import files
files.download(zip_path)

## Contrôle après génération

- Écouter chaque WAV et noter la prononciation, l'intelligibilité, les omissions et les artefacts.
- Comparer le son avec le texte wolof exact et sa source.
- Remplacer `unvalidated` par un statut de validation dans une copie de travail uniquement après écoute par un locuteur compétent.
- Ne pas intégrer automatiquement les WAV au dépôt ou à l'application. Les fichiers générés et le modèle ont des droits/conditions distincts : vérifier les conditions applicables avant redistribution.